# Model Training and Backtesting

In this notebook, we will train our Graph Convolutional Network (GCN) comparing it against a standard Multi-Layer Perceptron (MLP) baseline to prove the existence of *Contagion Alpha*.

### Why this works:
Traditional models treat companies independently. We are using PyTorch Geometric to pass momentum features through the supply chain/correlation graph, allowing the neural network to "see" upstream and downstream shocks.

In [ ]:
import torch
import pandas as pd
import numpy as np
import sys
sys.path.append('..')

from models.supply_chain_gcn import SupplyChainGCN, BaselineMLP
from src.backtester import backtest_strategy, calculate_metrics

# Set seed for reproducibility
torch.manual_seed(42)

## 1. Data Loader Mock
In a live scenario, this would load data generated from `data_loader.py` and `build_graph.py`. Here we establish the PyTorch tensors required.

In [ ]:
     # === load processed data from earlier pipeline ===
# Paths relative to this notebook file
import os
BASE_DIR = os.path.abspath(os.path.dirname(__file__))
DATA_PROCESSED = os.path.join(BASE_DIR, 'data', 'processed')

# reads returns and adjacency matrix
returns_5d = pd.read_csv(os.path.join(DATA_PROCESSED, 'returns_5d.csv'),
                         index_col=0, parse_dates=True)
returns_1d = pd.read_csv(os.path.join(DATA_PROCESSED, 'returns_1d.csv'),
                         index_col=0, parse_dates=True)
adj_matrix = pd.read_csv(os.path.join(DATA_PROCESSED, 'adjacency_matrix.csv'),
                         index_col=0)

# Basic sanity checks
if returns_5d.shape[1] != adj_matrix.shape[0] or adj_matrix.shape[0] != adj_matrix.shape[1]:
    raise ValueError('Number of tickers in returns and adjacency matrix must match')

# convert adjacency to edge_index
edges = np.vstack(np.nonzero(adj_matrix.values))
edge_index = torch.tensor(edges, dtype=torch.long)

# Dataset dimensions
NUM_NODES = adj_matrix.shape[0]
NUM_FEATURES = 1  # we only use 5-day return as a feature for now

# convert returns tables to tensors [T, N]
x_all = torch.tensor(returns_5d.values, dtype=torch.float)
y_all = torch.tensor(returns_1d.values, dtype=torch.float)

# drop any time steps where features or targets contain NaNs
valid_mask = (~torch.isnan(x_all).any(dim=1)) & (~torch.isnan(y_all).any(dim=1))
x_all = x_all[valid_mask]
y_all = y_all[valid_mask]

print(f"Loaded data with {x_all.shape[0]} dates, {NUM_NODES} assets")
print(f"Adjacency edges: {edge_index.shape}")
print(f"After dropping NaNs, {x_all.shape[0]} usable time steps")

## 2. Initialize Models
We instantiate both our GCN (which uses the graph) and our MLP (which ignores the graph).

In [ ]:
hidden_channels = 16

# Model 1: The Graph Convolutional Network
gcn_model = SupplyChainGCN(num_node_features=NUM_FEATURES, hidden_channels=hidden_channels)

# Model 2: The Baseline Multi-Layer Perceptron (No graph)
mlp_model = BaselineMLP(num_node_features=NUM_FEATURES, hidden_channels=hidden_channels)

optimizer_gcn = torch.optim.Adam(gcn_model.parameters(), lr=0.01)
optimizer_mlp = torch.optim.Adam(mlp_model.parameters(), lr=0.01)
loss_fn = torch.nn.MSELoss() # Mean Squared Error since predicting returns is regression


## 3. Training Loop Validation
Running a quick forward pass to ensure the architecture compiles and gradients flow.

In [ ]:
def train_step(model, optimizer, x, y_true, use_graph=True):
    model.train()
    optimizer.zero_grad()
    
    if use_graph:
        out = model(x, edge_index)
    else:
        out = model(x)  # MLP has no edge_index
    
    loss = loss_fn(out, y_true)
    loss.backward()
    optimizer.step()
    
    return loss.item()

# training loop over time-steps using real returns
EPOCHS = 5

print("Starting training on real data...")
for epoch in range(1, EPOCHS + 1):
    total_gcn_loss = 0.0
    total_mlp_loss = 0.0
    count = 0

    # we use pairs (t, t+1) where features at t predict return at t+1
    for t in range(x_all.shape[0] - 1):
        x = x_all[t].view(NUM_NODES, NUM_FEATURES)
        y_true = y_all[t + 1].view(NUM_NODES, 1)

        gcn_loss = train_step(gcn_model, optimizer_gcn, x, y_true, use_graph=True)
        mlp_loss = train_step(mlp_model, optimizer_mlp, x, y_true, use_graph=False)

        total_gcn_loss += gcn_loss
        total_mlp_loss += mlp_loss
        count += 1

    print(f"Epoch {epoch:03d} | GCN Avg Loss: {total_gcn_loss / count:.4f} | "
          f"MLP Avg Loss: {total_mlp_loss / count:.4f}")

## 4. Strategy Backtesting Engine
Once training is complete, the model outputs predictions for Out-Of-Sample data. We run those predictions through our Vectorized Backtester.

In [ ]:
# After training we can backtest over the same window
predictions = []
actuals = []
for t in range(x_all.shape[0] - 1):
    x = x_all[t].view(NUM_NODES, NUM_FEATURES)
    preds = gcn_model(x, edge_index).detach().numpy().flatten()
    predictions.append(preds)
    actuals.append(y_all[t + 1].numpy().flatten())

predictions = np.stack(predictions)
actuals = np.stack(actuals)

returns = backtest_strategy(predictions, actuals, quantile=0.9)
metrics = calculate_metrics(returns)